# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 02 — Model Selection

---

### Purpose
Justify the selection of the four classical ML classifiers used in this framework.
This notebook contains **no training code** — it is a scientific rationale document
that explains why each model was chosen, its strengths, weaknesses, and expected
behaviour on LLM fingerprinting tasks.

### Notebook Outline
1. Selection Criteria
2. Model Overview
3. Logistic Regression
4. Linear SVM
5. Random Forest
6. XGBoost
7. Feature × Model Compatibility Matrix
8. Decision Summary

---

## 1. Selection Criteria

The models were selected based on the following criteria:

| Criterion | Rationale |
|---|---|
| **Established baselines** | Models with well-understood behaviour in text classification |
| **Interpretability** | At least some models must support feature importance analysis |
| **Scalability** | Must handle sparse matrices with 30,000–50,000 dimensions |
| **Speed** | Must be trainable in reasonable time without GPU |
| **Diversity** | Cover linear, ensemble, and boosting paradigms |
| **Reproducibility** | All support `random_state` parameter for deterministic results |

---

## 2. Model Overview

| Model | Paradigm | Feature Compatibility | Interpretable |
|---|---|---|---|
| Logistic Regression | Linear probabilistic | Sparse (TF-IDF, Char) | ✅ Coefficients |
| Linear SVM | Linear margin-based | Sparse (TF-IDF, Char) | ✅ Weight vectors |
| Random Forest | Non-linear ensemble | Dense (Style, Emb) | ✅ Gini importance |
| XGBoost | Gradient boosting | Dense + Tabular | ✅ Gain importance |

---

## 3. Logistic Regression

### Description
Logistic Regression is a **linear probabilistic classifier** that models the
posterior probability of each class using a softmax function over a weighted
sum of input features.

### Mathematical Formulation
For K classes, the probability of class k is:
$$P(y=k \mid \mathbf{x}) = \frac{\exp(\mathbf{w}_k^\top \mathbf{x} + b_k)}{\sum_{j=1}^{K} \exp(\mathbf{w}_j^\top \mathbf{x} + b_j)}$$

The regularisation parameter **C** controls the trade-off between
fitting the training data (low C → strong regularisation) and
minimising training error (high C → weak regularisation).

### Strengths for LLM Fingerprinting
- ✅ **Probabilistic output**: Provides well-calibrated class probabilities
- ✅ **Sparse matrix efficiency**: Natively handles 50,000-dimensional TF-IDF matrices
- ✅ **Fast training**: Typically converges in seconds to minutes
- ✅ **Interpretable weights**: Coefficient inspection reveals which words/n-grams discriminate models
- ✅ **Excellent baseline**: Strong empirical results on text classification tasks
- ✅ **Multi-class support**: `lbfgs` and `saga` solvers support multinomial LR natively

### Weaknesses
- ❌ **Linear boundary**: Cannot model non-linear feature interactions
- ❌ **Feature independence assumption**: Ignores correlations between features
- ❌ **Less effective on dense embeddings**: Embeddings encode complex geometry that linear models may not fully exploit

### Expected Behaviour on LLM Fingerprinting
- **Best on**: TF-IDF word and character features
- **Competitive with**: Linear SVM
- **Likely outperformed by**: XGBoost on stylometric features
- **Key insight**: If LLMs are linearly separable in TF-IDF space (which is common
  in text classification), Logistic Regression may be the best or second-best overall model

### Hyperparameter Sensitivity
- **C**: Most sensitive. A grid search over [0.01, 0.1, 1.0, 10.0, 100.0] is recommended
- **solver**: `lbfgs` for multi-class; `saga` for very large datasets
- **class_weight**: Use `'balanced'` to handle any class imbalance in the dataset

---

## 4. Linear SVM

### Description
Linear Support Vector Machine (LinearSVC) finds the **maximum-margin hyperplane**
that separates classes in high-dimensional feature space.

### Mathematical Formulation
The primal optimisation objective is:
$$\min_{\mathbf{w}, b} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i} \ell(y_i (\mathbf{w}^\top \mathbf{x}_i + b))$$

where $\ell$ is the hinge loss. The decision boundary maximises the **margin**
between the closest training examples (support vectors) of each class.

### Strengths for LLM Fingerprinting
- ✅ **State-of-the-art on text**: Linear SVMs are historically among the best text classifiers
- ✅ **Sparse efficiency**: Highly optimised for CSR sparse matrices
- ✅ **Regularisation via C**: Prevents overfitting on high-dimensional feature spaces
- ✅ **Maximum margin**: Generalisable decision boundary
- ✅ **Fast inference**: Linear time prediction

### Weaknesses
- ❌ **No native probabilities**: LinearSVC does not output probabilities.
  We use `CalibratedClassifierCV` (Platt scaling) to add probability estimates.
- ❌ **Hard margin formulation**: Sensitive to outliers if C is too large
- ❌ **Binary in core**: Multi-class is implemented via one-vs-rest, which can
  produce inconsistent confidence scores across classes

### Expected Behaviour
- **Best on**: High-dimensional TF-IDF and Char N-Gram features
- **Competitive**: Often equals or beats Logistic Regression on sparse features
- **Likely weakness**: Dense stylometric features where non-linear models excel

### Implementation Note
In this project, LinearSVC is wrapped in `CalibratedClassifierCV` using 3-fold
cross-validation with sigmoid calibration. This enables `predict_proba` for
ROC-AUC computation and consistent evaluation with other models.

---

## 5. Random Forest

### Description
Random Forest is a **bagging ensemble** of decision trees where each tree is
trained on a bootstrap sample of the training data, and each split considers
a random subset of features.

### Why Ensembles?
Individual decision trees have **high variance** (overfit). By averaging predictions
of many decorrelated trees, Random Forest achieves **low variance** while maintaining
**low bias**. This is the **bias-variance tradeoff** in practice.

### Strengths for LLM Fingerprinting
- ✅ **Non-linear boundaries**: Captures complex feature interactions
- ✅ **No feature scaling**: Works equally well on raw stylometric and embedding features
- ✅ **Feature importance**: Gini-based importance reveals discriminative stylometric features
- ✅ **Robustness**: Tolerant to noisy, correlated features
- ✅ **Out-of-bag score**: Built-in validation estimate without a separate validation set
- ✅ **Parallelism**: `n_jobs=-1` enables full CPU parallelism

### Weaknesses
- ❌ **Memory intensive**: Storing 300 deep trees requires significant RAM
- ❌ **Slow on sparse matrices**: Sparse data structures are not efficiently exploited
- ❌ **Slower than linear models**: Training is much slower than LR or SVM
- ❌ **Limited extrapolation**: Cannot predict outside the range of training labels

### Expected Behaviour
- **Best on**: Dense stylometric and embedding features
- **Weaker on**: High-dimensional sparse TF-IDF (memory and time constraints)
- **Key strength**: Stylometric feature importance analysis — a key scientific insight

---

## 6. XGBoost

### Description
XGBoost (eXtreme Gradient Boosting) is a **sequential gradient boosting** algorithm
where each new tree corrects the residual errors of all previous trees.
It is widely considered the leading approach for structured/tabular data.

### Why Gradient Boosting?
Unlike Random Forest (parallel trees), XGBoost builds trees **sequentially**,
minimising a differentiable loss function using gradient descent in function space.
Combined with L1/L2 regularisation, it achieves excellent generalisation.

### Strengths for LLM Fingerprinting
- ✅ **Top performance on tabular data**: Consistently wins ML competitions
- ✅ **Built-in regularisation**: L1 (alpha) and L2 (lambda) prevent overfitting
- ✅ **Handles missing values**: Native support via optimal split direction
- ✅ **Feature importance**: Three importance types (gain, weight, cover)
- ✅ **Efficient implementation**: Column-block data structure for fast split finding
- ✅ **Early stopping**: Prevents overfitting during training

### Weaknesses
- ❌ **Hyperparameter sensitivity**: More hyperparameters than other models
- ❌ **Less effective on extreme sparsity**: Linear models often outperform on very sparse TF-IDF
- ❌ **Slower training**: Especially with many trees and deep max_depth
- ❌ **Harder to interpret**: Ensemble of hundreds of trees is less interpretable than a single LR model

### Expected Behaviour
- **Best on**: Stylometric and embedding features (dense, structured)
- **Competitive on**: Char N-Gram features
- **Likely weaker than SVM/LR on**: Very high-dimensional TF-IDF (50,000 dims)

---

## 7. Feature × Model Compatibility Matrix

| Feature Set | Logistic Regression | Linear SVM | Random Forest | XGBoost |
|---|:---:|:---:|:---:|:---:|
| TF-IDF Word (50k sparse) | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐ |
| TF-IDF Char (30k sparse) | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐ |
| Char N-Grams (combined) | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |
| Stylometric (~30 dense) | ⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| Embeddings (384 dense) | ⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |

⭐⭐⭐ = Excellent expected fit  
⭐⭐ = Good expected fit  
⭐ = Acceptable but not ideal

---

## 8. Decision Summary

### Why These Four Models?

1. **Coverage of paradigms**: Linear models (LR, SVM), ensemble (RF), boosting (XGB)
2. **Coverage of feature types**: Each model excels on different feature spaces
3. **Interpretability**: All four provide feature importance or coefficient inspection
4. **Reproducibility**: All support `random_state` and `sklearn`-compatible API
5. **Research validity**: These four models represent the current standard baselines
   for text classification in the NLP literature

### Scientific Hypotheses to Test

| Hypothesis | Test |
|---|---|
| Linear models excel on TF-IDF | Compare LR vs RF on TF-IDF features |
| Stylometric features favour tree models | Compare RF/XGB vs LR on style features |
| Fingerprint-preserving pipeline outperforms traditional | Compare Pipeline A vs B F1 scores |
| Embeddings are competitive with sparse methods | Compare embedding vs TF-IDF F1 |

---

> **Next**: Notebooks 03–06 train and evaluate each model independently.
> Notebook 08 provides the full cross-model comparison.

*Fingerprint Project — Model Selection — Complete*